# Day3_03C_CrewAI_Sequential_vs_Hierarchical

## CrewAI - Process Types

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 3 - CrewAI  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand the difference between **Sequential** and **Hierarchical** CrewAI processes.
- Explain when to use sequential execution.
- Explain when to use hierarchical execution.
- Build a simple sequential Crew.
- Build a manager-led hierarchical Crew.
- Relate both patterns to DSU and enterprise workflows.

---

## Previous Notebook Recap

In the previous notebook, we learned **context chaining**.

We saw that task outputs can flow like this:

```text
Research Task
   ↓
Write Task
   ↓
Review Task
```

Now we will learn how CrewAI controls the overall workflow.

This is called the **process type**.


# 1. Why Process Type Matters

Imagine preparing an FDP session.

There are two ways to organize the work.

## Option 1: Fixed Order

```text
Research
   ↓
Write
   ↓
Review
```

Everyone follows a fixed sequence.

This is called a **Sequential Process**.

---

## Option 2: Manager-Led Work

```text
Manager
   ├── Assigns research
   ├── Assigns writing
   ├── Asks for review
   └── Coordinates final output
```

A manager decides how work should move.

This is called a **Hierarchical Process**.


# 2. Sequential vs Hierarchical

| Process Type | Simple Meaning | Best For |
|---|---|---|
| Sequential | Tasks run in listed order | Simple, fixed workflows |
| Hierarchical | Manager Agent coordinates work | Complex, dynamic workflows |

---

## Sequential Process

Use when:

- Task order is known
- Output of one task feeds the next
- Workflow is predictable
- Debugging should be easy

Example:

```text
Research → Write → Review
```

---

## Hierarchical Process

Use when:

- A manager should coordinate
- Tasks may need delegation
- Work may need refinement
- The flow is more dynamic

Example:

```text
Manager → Assigns work → Reviews → Reassigns if needed
```


# Pause & Reflect

Ask the class:

> If you are preparing a simple assignment question paper, would you need a manager Agent?

Probably not.

Sequential is enough.

But if you are preparing an entire 5-day FDP with content, labs, quizzes, review, and improvement, a manager Agent becomes useful.


# 3. Setup

Install dependencies if needed:

```bash
pip install crewai crewai-tools python-dotenv
```

We will use the same `.env` structure as the previous notebooks.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from crewai import Agent, Task, Crew, Process

# 4. Load Environment Variables

In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 5. Create Agents for Sequential Workflow

We will first build a simple sequential Crew for preparing a mini lecture module.

Agents:

1. Researcher
2. Writer
3. Reviewer


In [ ]:
researcher = Agent(
    role="AI Topic Researcher",
    goal="Research simple and practical points for engineering faculty",
    backstory="""
    You are an education-focused AI researcher. You simplify technical topics
    and identify useful classroom examples.
    """,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Lecture Module Writer",
    goal="Convert research notes into a short teaching module",
    backstory="""
    You write faculty-friendly teaching modules using simple language,
    examples, and classroom flow.
    """,
    verbose=True,
    allow_delegation=False,
)

reviewer = Agent(
    role="Academic Reviewer",
    goal="Review the teaching module for clarity and classroom usefulness",
    backstory="""
    You are a senior faculty reviewer. You improve teaching material
    and ensure it is useful for engineering professors.
    """,
    verbose=True,
    allow_delegation=False,
)

# 6. Define Sequential Tasks

In a sequential process, tasks are executed in the order we list them.

```text
Research → Write → Review
```


In [ ]:
research_task = Task(
    description="""
    Research the topic: CrewAI process types.

    Explain:
    1. What sequential process means
    2. What hierarchical process means
    3. One classroom analogy
    4. One enterprise example
    """,
    expected_output="A concise research brief with four sections.",
    agent=researcher,
)

write_task = Task(
    description="""
    Using the research brief, create a short 10-minute teaching module
    for engineering faculty.
    """,
    expected_output="A simple teaching module with definition, analogy, and example.",
    agent=writer,
    context=[research_task],
)

review_task = Task(
    description="""
    Review and improve the teaching module.

    Make it:
    - simpler
    - classroom-friendly
    - suitable for faculty with minimal coding background
    """,
    expected_output="An improved teaching module ready for classroom delivery.",
    agent=reviewer,
    context=[research_task, write_task],
)

# 7. Create and Run Sequential Crew

The key line is:

```python
process=Process.sequential
```


In [ ]:
sequential_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, write_task, review_task],
    process=Process.sequential,
    verbose=True,
)

sequential_result = sequential_crew.kickoff()

print(sequential_result)

## What did we just learn?

Sequential process is easy to understand.

The flow is fixed:

```text
Task 1 → Task 2 → Task 3
```

This is ideal for beginner-friendly teaching notebooks.


# 8. When Sequential Works Best

Use sequential process when the workflow is predictable.

Examples:

## Academic

```text
Research notes → Lesson plan → Review
```

## Software

```text
Requirement → Design → Test cases
```

## Enterprise

```text
Customer issue → Investigation → Response draft
```

Sequential process is simple, traceable, and easy to debug.


# 9. Hierarchical Process Concept

Now let us understand hierarchical process.

In a hierarchical process, a manager Agent coordinates work.

```text
Manager Agent
   ├── Researcher
   ├── Writer
   └── Reviewer
```

The manager can decide how to organize the work.

This is useful when the workflow is less rigid.


# 10. Create a Manager Agent

The manager Agent should understand the final goal and coordinate the team.


In [ ]:
manager = Agent(
    role="FDP Content Manager",
    goal="Coordinate the team to produce a high-quality faculty training module",
    backstory="""
    You are an experienced FDP coordinator. You know how to divide work among
    researchers, writers, and reviewers. You ensure the final output is clear,
    practical, and classroom-ready.
    """,
    verbose=True,
    allow_delegation=True,
)

# 11. Define a Manager-Led Task

For hierarchical process, we can define a broader task and allow the manager to coordinate the work.

Note:

Depending on your CrewAI version, hierarchical process may require manager configuration such as `manager_agent`.


In [ ]:
manager_task = Task(
    description="""
    Prepare a complete 10-minute faculty teaching module on:

    Topic: Sequential vs Hierarchical process in CrewAI

    The final output should include:
    1. Simple explanation
    2. Difference table
    3. DSU classroom example
    4. Enterprise example
    5. Key takeaways
    """,
    expected_output="A complete faculty-ready teaching module on CrewAI process types.",
    agent=manager,
)

# 12. Create Hierarchical Crew

The key line is:

```python
process=Process.hierarchical
```

We also provide a manager Agent.

If your installed CrewAI version behaves differently, use the sequential version for classroom demo and explain hierarchical conceptually.


In [ ]:
hierarchical_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[manager_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# 13. Run Hierarchical Crew

This run may take longer because the manager coordinates the work.

If your local CrewAI version gives a parameter-related error, check your CrewAI version and documentation.
For classroom safety, keep the sequential demo as the primary working demo.


In [ ]:
hierarchical_result = hierarchical_crew.kickoff()

print(hierarchical_result)

# 14. What Did We Learn from Hierarchical Process?

Hierarchical process is useful when:

- Work is complex
- Manager coordination is needed
- Tasks may need delegation
- Output requires synthesis
- The workflow is not strictly fixed

But it may also be:

- Slower
- More expensive
- Harder to debug
- Dependent on CrewAI version behavior

So use it when the problem justifies it.


# 15. Sequential vs Hierarchical Decision Guide

| Situation | Recommended Process |
|---|---|
| Research → Write → Review | Sequential |
| Simple lecture module | Sequential |
| Assignment creation | Sequential |
| Multi-team FDP planning | Hierarchical |
| Complex enterprise investigation | Hierarchical |
| Dynamic task allocation | Hierarchical |
| Beginner classroom demo | Sequential first |

---

## Simple Rule

If the path is fixed, use **Sequential**.

If coordination is required, use **Hierarchical**.


# 16. DSU Example

## Sequential

Preparing a lesson:

```text
Research topic
   ↓
Write module
   ↓
Review module
```

Best process: **Sequential**

---

## Hierarchical

Preparing a 5-day FDP:

```text
FDP Coordinator
   ├── Day 1 Content Team
   ├── Day 2 Demo Team
   ├── Day 3 Agent Team
   ├── Assessment Team
   └── Review Team
```

Best process: **Hierarchical**


# 17. Enterprise Example

## Sequential

Creating a customer response:

```text
Read complaint
   ↓
Draft response
   ↓
Review tone
```

Best process: **Sequential**

---

## Hierarchical

Investigating a production incident:

```text
Incident Manager
   ├── Log Analysis Agent
   ├── Metrics Agent
   ├── Deployment Agent
   ├── Security Agent
   └── RCA Writer Agent
```

Best process: **Hierarchical**


# 18. Classroom Exercise

For each scenario, choose Sequential or Hierarchical.

| Scenario | Process |
|---|---|
| Create a quiz from a lesson plan | ? |
| Prepare an entire FDP program | ? |
| Review a student assignment | ? |
| Investigate a production outage | ? |
| Create a 10-minute lecture note | ? |
| Coordinate admission, hostel, fee, and course counselling | ? |


# 19. Hands-on Exercise

Create a sequential Crew for:

> Preparing a student placement training module

Tasks:

1. Research placement skills
2. Create training module
3. Review and improve module

Use:

- Placement Researcher
- Training Content Writer
- Placement Reviewer


In [ ]:
# Exercise Starter Code

placement_researcher = Agent(
    role="Placement Skills Researcher",
    goal="Research important placement preparation skills for engineering students",
    backstory="You understand campus placement requirements and student readiness.",
    verbose=True,
    allow_delegation=False,
)

placement_writer = Agent(
    role="Placement Training Writer",
    goal="Create a simple training module for students",
    backstory="You write practical student training content.",
    verbose=True,
    allow_delegation=False,
)

placement_reviewer = Agent(
    role="Placement Training Reviewer",
    goal="Review placement training material for usefulness and clarity",
    backstory="You are a placement officer who reviews training content.",
    verbose=True,
    allow_delegation=False,
)

# TODO:
# 1. Create research task
# 2. Create writing task with context
# 3. Create review task with context
# 4. Create sequential Crew
# 5. Run kickoff()


# 20. Challenge Exercise

Design a hierarchical Crew for:

> Planning a 1-day Agentic AI workshop

Manager:

- Workshop Coordinator

Specialists:

- Concept Planner
- Demo Planner
- Exercise Designer
- Assessment Reviewer

Final output:

- Workshop agenda
- Session-wise content
- Demo plan
- Exercise plan
- Assessment plan


# 21. Common Mistakes

Avoid these mistakes:

1. Using hierarchical process for very simple workflows.
2. Forgetting that hierarchical process may take more time.
3. Not giving the manager a clear goal.
4. Creating specialists with overlapping responsibilities.
5. Not using sequential process for fixed pipelines.
6. Ignoring CrewAI version differences.
7. Assuming hierarchical is always better.

---

## Teaching Tip

Start with sequential.

Once participants understand Agent, Task, Crew, and Context, then introduce hierarchical as an advanced orchestration pattern.


# 22. Key Takeaways

In this notebook, we learned:

- CrewAI supports different process types.
- Sequential process runs tasks in listed order.
- Hierarchical process uses a manager Agent to coordinate work.
- Sequential is best for predictable workflows.
- Hierarchical is best for complex, manager-led workflows.
- For classroom teaching, sequential is easier to demonstrate first.
- Choose the process based on workflow complexity.

---

## Final Mental Model

```text
Sequential:
Task 1 → Task 2 → Task 3

Hierarchical:
Manager
 ├── Agent 1
 ├── Agent 2
 └── Agent 3
```

Use the simplest process that solves the problem.
